# Genex Therapy Agents — Hybrid CDC + Bright Futures (v2)

This version adds:
- **safe output saving** to `outputs/therapy_agents/`
- a **Bright Futures normalized reference table**
- a **diagnosis/concern overlay** so older children with ADHD are not automatically treated as "no support needed"
- a more explicit distinction between:
  - **milestone delay** (CDC-style)
  - **functioning support needs** (Bright Futures-style)

Use this when a child may not have a large milestone delay but still needs help with:
- attention / concentration
- routines / school functioning
- peer interaction / emotional regulation

In [ ]:
%pip install pandas openpyxl openai python-dotenv ipython

In [ ]:
import os
import json
import re
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown

try:
    from openai import OpenAI
except Exception:
    OpenAI = None

load_dotenv()

# -----------------------------
# Project paths
# -----------------------------
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "therapy_agents"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

# -----------------------------
# Domain config
# -----------------------------
DOMAIN_CONFIG = {
    "movement_and_physical": {"display": "Movement / Physical", "short": "motor"},
    "social_and_emotial": {"display": "Social / Emotional", "short": "social_emotional"},
    "language_and_communication": {"display": "Language / Communication", "short": "language_communication"},
    "cognitive": {"display": "Cognitive / Adaptive", "short": "cognitive"},
}

ALIAS_TO_CATEGORY = {
    "movement and physical": "movement_and_physical",
    "motor": "movement_and_physical",
    "social and emotial": "social_and_emotial",
    "social and emotional": "social_and_emotial",
    "social": "social_and_emotial",
    "language and communication": "language_and_communication",
    "language": "language_and_communication",
    "speech": "language_and_communication",
    "speech and language": "language_and_communication",
    "cognitive": "cognitive",
    "adaptive": "cognitive",
}

ANSWER_SCORES = {
    "yes": 1.0,
    "sometimes": 0.45,
    "with_help": 0.2,
    "no": 0.0,
    "not_sure": 0.1,
}

VALID_ANSWERS = set(ANSWER_SCORES.keys())

def normalize_answer(answer_text: str) -> str:
    t = str(answer_text).strip().lower().replace(" ", "_")
    if t in VALID_ANSWERS:
        return t
    if t in {"notsure", "unsure", "maybe"}:
        return "not_sure"
    return "not_sure"

# -----------------------------
# CDC loading
# -----------------------------
def find_cdc_file(filename="milestone-cdc-table.xlsx"):
    search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for root in search_roots:
        candidate = root / filename
        if candidate.exists():
            return candidate.resolve()
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0].resolve()
    raise FileNotFoundError(f"Could not find {filename}.")

def load_cdc_table(path=None):
    path = Path(path) if path else find_cdc_file()
    df = pd.read_excel(path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    df = df.rename(columns={"category ": "category", "milestone ": "milestone"})
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    df["milestone"] = df["milestone"].astype(str).str.strip()
    df["months"] = pd.to_numeric(df["months"], errors="coerce")
    df = df.dropna(subset=["months", "category", "milestone"]).copy()
    df["months"] = df["months"].astype(int)
    df["category_key"] = df["category"].map(lambda x: ALIAS_TO_CATEGORY.get(x, x.replace(" ", "_")))
    df["question_id"] = [f"{row.category_key}_{row.months}_{i}" for i, row in enumerate(df.itertuples(), start=1)]
    return df, path

cdc_df, cdc_path = load_cdc_table()
CDC_AGES = sorted(cdc_df["months"].dropna().unique().tolist())

def get_category_questions(category_key: str, min_months: int, max_months: int) -> pd.DataFrame:
    subset = cdc_df[
        (cdc_df["category_key"] == category_key)
        & (cdc_df["months"] >= min_months)
        & (cdc_df["months"] <= max_months)
    ].sort_values(["months", "milestone"])
    return subset.copy()

# -----------------------------
# Bright Futures normalized table
# -----------------------------
# This is a compact normalized reference layer based on the uploaded Bright Futures handout.
BF_ROWS = [
    # 4 years
    {"age_label":"4 Years","age_min":48,"age_max":59,"bf_domain":"school_readiness","mapped_domain":"cognitive","text":"Language understanding and fluency; feelings; opportunities to socialize with other children; readiness for structured learning experiences; preschool."},
    {"age_label":"4 Years","age_min":48,"age_max":59,"bf_domain":"media_use","mapped_domain":"social_and_emotial","text":"Limits on use; promoting physical activity and safe play."},
    # 5 years
    {"age_label":"5 Years","age_min":60,"age_max":71,"bf_domain":"development_and_mental_health","mapped_domain":"social_and_emotial","text":"Family rules and routines, concern for others, patience and control over anger."},
    {"age_label":"5 Years","age_min":60,"age_max":71,"bf_domain":"school","mapped_domain":"cognitive","text":"Readiness, established routines, school attendance, and friends; after-school care and parent-teacher communication."},
    {"age_label":"5 Years","age_min":60,"age_max":71,"bf_domain":"physical_growth_and_development","mapped_domain":"movement_and_physical","text":"Oral health, nutrition, physical activity."},
    # 6 years
    {"age_label":"6 Years","age_min":72,"age_max":83,"bf_domain":"development_and_mental_health","mapped_domain":"social_and_emotial","text":"Family rules and routines, concern for others, patience and control over anger."},
    {"age_label":"6 Years","age_min":72,"age_max":83,"bf_domain":"school","mapped_domain":"cognitive","text":"Readiness, established routines, school attendance, and friends; after-school care and parent-teacher communication."},
    {"age_label":"6 Years","age_min":72,"age_max":83,"bf_domain":"physical_growth_and_development","mapped_domain":"movement_and_physical","text":"Oral health, nutrition, physical activity."},
    # 7-8 years
    {"age_label":"7-8 Years","age_min":84,"age_max":107,"bf_domain":"development_and_mental_health","mapped_domain":"social_and_emotial","text":"Independence; rules and consequences; temper problems and conflict resolution; pubertal development."},
    {"age_label":"7-8 Years","age_min":84,"age_max":107,"bf_domain":"school","mapped_domain":"cognitive","text":"Adaptation to school; school problems (behavior or learning issues); school performance and progress; school attendance; IEP or special education services."},
    {"age_label":"7-8 Years","age_min":84,"age_max":107,"bf_domain":"physical_growth_and_development","mapped_domain":"movement_and_physical","text":"Oral health, nutrition, physical activity."},
    # 9-10 years
    {"age_label":"9-10 Years","age_min":108,"age_max":131,"bf_domain":"development_and_mental_health","mapped_domain":"social_and_emotial","text":"Temper problems, setting reasonable limits, friends, sexuality."},
    {"age_label":"9-10 Years","age_min":108,"age_max":131,"bf_domain":"school","mapped_domain":"cognitive","text":"School attendance, school problems, school performance and progress, transitions, co-occurrence of middle school and pubertal transitions."},
    {"age_label":"9-10 Years","age_min":108,"age_max":131,"bf_domain":"physical_growth_and_development","mapped_domain":"movement_and_physical","text":"Oral health, nutrition, physical activity."},
]

bright_futures_df = pd.DataFrame(BF_ROWS)

def bf_rows_for_age(chronological_months: int) -> pd.DataFrame:
    return bright_futures_df[
        (bright_futures_df["age_min"] <= chronological_months)
        & (bright_futures_df["age_max"] >= chronological_months)
    ].copy()

# -----------------------------
# Agent state
# -----------------------------
def new_state():
    return {
        "child": {},
        "delay_estimates": {},
        "qna": {},
        "dev_age": {},
        "bf_overlay": {},
        "support_recommendations": {},
    }

def print_banner(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

def intake_agent_live(state):
    print_banner("INTAKE AGENT")
    name = input("What is your child's name? ").strip()
    age_years = float(input("How old is your child in years? ").strip())
    diagnosis = input("Any diagnosis / condition? (type 'none' if none) ").strip()
    concern = input("What are your main concerns right now? ").strip()

    state["child"] = {
        "name": name,
        "age_years": age_years,
        "chronological_months": int(round(age_years * 12)),
        "diagnosis": diagnosis,
        "concern": concern,
    }
    return state["child"]

# -----------------------------
# Delay estimate agent
# -----------------------------
def estimate_delay_gpt(diagnosis, concern, chronological_months, category_key, model="gpt-4o-mini"):
    if OpenAI is None or not os.environ.get("OPENAI_API_KEY"):
        # fallback starting estimate; do not overstate
        base = 6 if category_key in {"social_and_emotial", "cognitive"} else 3
        return {"delay_months": base, "reason": "Fallback starting estimate", "source": "fallback"}

    client = OpenAI()
    category_display = DOMAIN_CONFIG[category_key]["display"]

    prompt = f"""
You are helping with a NON-DIAGNOSTIC developmental intake.

Child:
- Chronological age in months: {chronological_months}
- Diagnosis / condition: {diagnosis}
- Parent concern: {concern}
- Domain: {category_display}

Task:
Estimate a conservative starting delay in months for this one domain to choose milestone questions.
Important:
- A child may have NO major milestone delay but still have functioning support needs.
- ADHD should not automatically be treated as a large motor or language delay.
- For ADHD, social/emotional and cognitive/school-functioning may still need support even if milestone delay is mild.

Return strict JSON:
{{
  "delay_months": <integer from 0 to chronological age>,
  "reason": "<one short sentence>"
}}
""".strip()

    try:
        resp = client.chat.completions.create(
            model=model,
            temperature=0.2,
            response_format={"type": "json_object"},
            messages=[{"role":"system","content":"Return strict JSON only."},
                      {"role":"user","content":prompt}],
        )
        parsed = json.loads(resp.choices[0].message.content)
        delay_months = int(max(0, min(int(parsed.get("delay_months", 6)), chronological_months)))
        return {"delay_months": delay_months, "reason": parsed.get("reason",""), "source":"openai"}
    except Exception as e:
        base = 6 if category_key in {"social_and_emotial", "cognitive"} else 3
        return {"delay_months": base, "reason": f"Fallback because OpenAI failed: {e}", "source":"fallback"}

def delay_agent_all_categories(state):
    print_banner("DELAY ESTIMATOR AGENT")
    child = state["child"]
    for category_key in DOMAIN_CONFIG:
        est = estimate_delay_gpt(
            diagnosis=child["diagnosis"],
            concern=child["concern"],
            chronological_months=child["chronological_months"],
            category_key=category_key,
        )
        state["delay_estimates"][category_key] = est
        print(f"{DOMAIN_CONFIG[category_key]['display']}: {est['delay_months']} month starting delay | {est['reason']}")
    return state["delay_estimates"]

# -----------------------------
# Milestone Q&A agent
# -----------------------------
def build_milestone_questions(state, category_key, window_months=12, max_questions=8):
    child = state["child"]
    chrono = min(child["chronological_months"], 60)
    delay_months = state["delay_estimates"].get(category_key, {}).get("delay_months", 6)

    approx_dev = max(2, chrono - delay_months)
    min_months = max(2, approx_dev - window_months // 2)
    max_months = min(60, approx_dev + window_months // 2)

    subset = get_category_questions(category_key, min_months=min_months, max_months=max_months)
    if subset.empty:
        subset = get_category_questions(category_key, min_months=min(CDC_AGES), max_months=max(CDC_AGES))

    questions = []
    for _, row in subset.iterrows():
        questions.append({
            "question_id": row["question_id"],
            "months": int(row["months"]),
            "milestone": row["milestone"],
            "question_text": f"Can {child['name']} {row['milestone']} right now? (yes / sometimes / with_help / no / not_sure)",
        })

    questions = sorted(questions, key=lambda x: (x["months"], x["milestone"]))
    return questions[:max_questions]

def compute_dev_age_from_answers(answers):
    if not answers:
        return 6
    answered_months = sorted(set(a["months"] for a in answers))
    yes_months = [a["months"] for a in answers if a["norm_answer"] == "yes"]
    partial_months = [a["months"] for a in answers if a["norm_answer"] in {"sometimes", "with_help", "not_sure"}]
    if yes_months:
        return int(max(yes_months))
    if partial_months:
        anchor = int(min(partial_months))
        lower_candidates = [m for m in answered_months if m < anchor]
        return int(max(lower_candidates)) if lower_candidates else int(anchor)
    return int(min(answered_months))

def qna_agent_live(state, category_key, ask_limit=5):
    print_banner(f"MILESTONE Q&A AGENT — {DOMAIN_CONFIG[category_key]['display']}")
    questions = build_milestone_questions(state, category_key)
    asked = []

    for q in questions[:ask_limit]:
        ans = input(q["question_text"] + "\n").strip()
        norm = normalize_answer(ans)
        asked.append({
            "question_id": q["question_id"],
            "months": q["months"],
            "milestone": q["milestone"],
            "raw_answer": ans,
            "norm_answer": norm,
            "score": ANSWER_SCORES[norm],
        })

    dev_age = compute_dev_age_from_answers(asked)
    state["qna"][category_key] = asked
    state["dev_age"][category_key] = dev_age

    chrono = state["child"]["chronological_months"]
    print(f"Estimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: {dev_age} months (chronological age {chrono} months)")
    return {"category": category_key, "dev_age_months": dev_age, "answers": asked}

# -----------------------------
# Bright Futures overlay agent
# -----------------------------
def compute_bf_overlay(state):
    """
    This agent catches needs that do not look like milestone delay,
    especially ADHD-like school / attention / social-emotional needs in older kids.
    """
    print_banner("BRIGHT FUTURES OVERLAY AGENT")
    child = state["child"]
    diagnosis = child["diagnosis"].lower()
    concern = child["concern"].lower()
    chrono = child["chronological_months"]

    age_rows = bf_rows_for_age(chrono)
    overlay = {k: {"flag": False, "reason": "", "reference_items": []} for k in DOMAIN_CONFIG}

    for category_key in DOMAIN_CONFIG:
        rows = age_rows[age_rows["mapped_domain"] == category_key]
        overlay[category_key]["reference_items"] = rows["text"].tolist()

    # ADHD / attention overlay
    attention_terms = ["adhd", "add", "attention", "focus", "concentration", "school", "peer", "friend", "behavior", "transition", "routine", "emotion", "anger"]
    has_attention_signal = any(term in diagnosis or term in concern for term in attention_terms)

    if has_attention_signal and chrono >= 60:
        overlay["social_and_emotial"]["flag"] = True
        overlay["social_and_emotial"]["reason"] = "Older child has ADHD / attention-social concerns, so social-emotional support may be needed even without major milestone delay."
        overlay["cognitive"]["flag"] = True
        overlay["cognitive"]["reason"] = "Older child has ADHD / school-attention concerns, so cognitive / adaptive / school-functioning support may be needed even without major milestone delay."

    state["bf_overlay"] = overlay
    return overlay

# -----------------------------
# Support decision
# -----------------------------
def no_special_support_needed(state, category_key):
    child = state["child"]
    chrono_for_compare = min(child["chronological_months"], 60)
    dev_age = state["dev_age"].get(category_key, chrono_for_compare)
    delay_est = state["delay_estimates"].get(category_key, {}).get("delay_months", 0)
    bf_flag = state.get("bf_overlay", {}).get(category_key, {}).get("flag", False)

    # If BF overlay says support is still needed, do not suppress the plan.
    if bf_flag:
        return False

    # Otherwise, only suppress when milestone delay is truly minimal.
    return (chrono_for_compare - dev_age) <= 6 and delay_est <= 6

def select_next_milestones(state, category_key, max_milestones=6):
    child = state["child"]
    dev_age = state["dev_age"].get(category_key, None)
    if dev_age is None:
        raise ValueError(f"No developmental age found for {category_key}.")

    if no_special_support_needed(state, category_key):
        return {
            "status": "no_special_support",
            "message": f"We do not think {child['name']} has a meaningful milestone delay in {DOMAIN_CONFIG[category_key]['display']} and may not need special support in this category right now.",
            "milestones": [],
        }

    min_m = dev_age + 1
    max_m = min(60, dev_age + 12)
    subset = get_category_questions(category_key, min_months=min_m, max_months=max_m)

    milestones = []
    for _, row in subset.iterrows():
        milestones.append({"months": int(row["months"]), "milestone": row["milestone"]})

    return {"status":"success","milestones": milestones[:max_milestones]}

# -----------------------------
# Support plan agent
# -----------------------------
def generate_support_plan(state, category_key, model="gpt-4o-mini"):
    child = state["child"]
    category_display = DOMAIN_CONFIG[category_key]["display"]
    next_steps = select_next_milestones(state, category_key)
    bf_overlay = state.get("bf_overlay", {}).get(category_key, {})

    if next_steps["status"] == "no_special_support":
        state["support_recommendations"][category_key] = {
            "status": "no_special_support",
            "summary": next_steps["message"],
            "plan": None,
        }
        return state["support_recommendations"][category_key]

    milestone_lines = "\n".join([f"- ({m['months']} months) {m['milestone']}" for m in next_steps["milestones"]]) or "- no specific CDC next-step milestones selected"
    bf_lines = "\n".join([f"- {x}" for x in bf_overlay.get("reference_items", [])]) or "- no Bright Futures age-reference items"
    overlay_reason = bf_overlay.get("reason", "")

    if OpenAI is None or not os.environ.get("OPENAI_API_KEY"):
        summary = f"Support suggested for {category_display}."
        if overlay_reason:
            summary += f" {overlay_reason}"
        state["support_recommendations"][category_key] = {
            "status": "fallback",
            "summary": summary,
            "plan": {
                "day_1":[f"Short home support activity for {category_display.lower()}"],
                "day_2":[f"Repeat with slight variation for {category_display.lower()}"],
                "day_3":[f"Practice during a familiar routine for {category_display.lower()}"],
                "day_4":[f"Use play or school-like practice for {category_display.lower()}"],
                "day_5":[f"Review progress and repeat the easiest activity for {category_display.lower()}"],
            }
        }
        return state["support_recommendations"][category_key]

    client = OpenAI()
    prompt = f"""
You are a pediatric support planner for families.

Child:
- Name: {child['name']}
- Chronological age: {child['age_years']} years ({child['chronological_months']} months)
- Diagnosis / condition: {child['diagnosis']}
- Parent concern: {child['concern']}
- Category: {category_display}

Estimated developmental age in this category: {state['dev_age'].get(category_key)} months

CDC next-step milestone ideas:
{milestone_lines}

Bright Futures age-reference items:
{bf_lines}

Extra context:
{overlay_reason if overlay_reason else "No extra overlay reason."}

Task:
Create a brief 5-day home support plan with 1-2 realistic items per day.
If the category has minimal milestone delay but still needs functioning support, make that clear.
Stay non-diagnostic and parent-friendly.
Return strict JSON only:
{{
  "summary": "...",
  "day_1": ["..."],
  "day_2": ["..."],
  "day_3": ["..."],
  "day_4": ["..."],
  "day_5": ["..."]
}}
""".strip()

    try:
        resp = client.chat.completions.create(
            model=model,
            temperature=0.3,
            response_format={"type":"json_object"},
            messages=[{"role":"system","content":"Return strict JSON only and stay non-diagnostic."},
                      {"role":"user","content":prompt}],
        )
        parsed = json.loads(resp.choices[0].message.content)
        state["support_recommendations"][category_key] = {"status":"success","summary": parsed.get("summary",""), "plan": parsed}
        return state["support_recommendations"][category_key]
    except Exception as e:
        summary = f"Support suggested for {category_display}."
        if overlay_reason:
            summary += f" {overlay_reason}"
        state["support_recommendations"][category_key] = {
            "status":"fallback",
            "summary": summary + f" Fallback used because OpenAI failed: {e}",
            "plan": {
                "day_1":[f"Short home support activity for {category_display.lower()}"],
                "day_2":[f"Repeat with slight variation for {category_display.lower()}"],
                "day_3":[f"Practice during a familiar routine for {category_display.lower()}"],
                "day_4":[f"Use play or school-like practice for {category_display.lower()}"],
                "day_5":[f"Review progress and repeat the easiest activity for {category_display.lower()}"],
            }
        }
        return state["support_recommendations"][category_key]

def summarizer_agent(state):
    child = state["child"]
    rows = []
    for category_key in DOMAIN_CONFIG:
        reco = state["support_recommendations"].get(category_key, {})
        bf_flag = state.get("bf_overlay", {}).get(category_key, {}).get("flag", False)
        rows.append({
            "child_name": child.get("name"),
            "chronological_age_months": child.get("chronological_months"),
            "diagnosis": child.get("diagnosis"),
            "category": DOMAIN_CONFIG[category_key]["display"],
            "estimated_delay_months": state["delay_estimates"].get(category_key, {}).get("delay_months"),
            "estimated_dev_age_months": state["dev_age"].get(category_key),
            "bright_futures_overlay_flag": bf_flag,
            "needs_special_support": False if reco.get("status") == "no_special_support" else True,
            "summary": reco.get("summary", ""),
        })
    return pd.DataFrame(rows)

def run_all_categories_live():
    state = new_state()
    intake_agent_live(state)
    delay_agent_all_categories(state)
    for category_key in DOMAIN_CONFIG:
        qna_agent_live(state, category_key)
    compute_bf_overlay(state)
    for category_key in DOMAIN_CONFIG:
        generate_support_plan(state, category_key)
    summary_df = summarizer_agent(state)
    return state, summary_df

In [ ]:
display(cdc_df.head())
display(bright_futures_df.head(10))

## Why ADHD was missed before

A 6-year-old with ADHD can still meet many early milestone-style items, so a pure CDC developmental-age logic may conclude "no meaningful milestone delay." But Bright Futures for ages 5–10 shifts attention toward school readiness, rules/routines, peer relationships, school problems, behavior, and emotional regulation rather than only early milestones. This is why older ADHD support can be real even when a child does not look developmentally delayed on a milestone table. fileciteturn12file6turn12file1turn12file2

In [ ]:
state, summary_df = run_all_categories_live()
summary_df

In [ ]:
# Save outputs outside the notebook folder
summary_path = OUTPUT_DIR / f"genex_hybrid_summary_{RUN_STAMP}.csv"
bf_ref_path = OUTPUT_DIR / f"bright_futures_reference_{RUN_STAMP}.csv"
state_path = OUTPUT_DIR / f"genex_hybrid_state_{RUN_STAMP}.json"

summary_df.to_csv(summary_path, index=False)
bright_futures_df.to_csv(bf_ref_path, index=False)

with open(state_path, "w", encoding="utf-8") as f:
    json.dump(state, f, indent=2)

print("Saved:")
print(summary_path)
print(bf_ref_path)
print(state_path)

In [ ]:
# Optional: inspect support details for one category
state["support_recommendations"]